# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prakharps1305-dev/flyrank/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
import os, sys, subprocess
if "google.colab" in sys.modules:
    if not os.path.isdir("flyrank-ml-internship-starter"):
        subprocess.run(["git","clone","--depth","1",
          "https://github.com/flyrank-bih/flyrank-ml-internship-starter"], check=True)
    os.chdir("flyrank-ml-internship-starter")
print(os.getcwd())

/content/flyrank-ml-internship-starter/flyrank-ml-internship-starter


## 1. My lane (or freestyle) and why

**Lane 4 — CTR / Engagement Opportunity Scoring.**

I picked this lane because its central move is one I want to get right: build an expected value conditional on a confounder, then rank by the gap. CTR is not comparable across search positions — in notebook 01 mean CTR fell from 0.355 on page 1 to 0.055 for deep results — so any "this page underperforms" claim that ignores position is measuring position, not performance. Comparing each page only against others in its own position tier, and ranking by the residual, forces me to confront that. The other lanes teach useful things too, but correlation summaries and precision@K I will pick up along the way regardless; the habit of asking "compared to what?" is the one I most want to build.

## 2. The question: decision, action, cost of a wrong call

**Question.** Among pages that already have meaningful search visibility, which ones capture fewer clicks than comparable pages at the same position tier, and therefore deserve a title, meta-description, or snippet review first?

**Unit of analysis.** One page (`content_id`). Not a client, not a day.

**Decision it improves.** A content team has limited review capacity — realistically a few dozen pages per cycle out of thousands. The decision is which pages enter that review queue, and in what order.

**Who acts.** An SEO or content editor, who rewrites the title/meta, improves intent match, or restructures the snippet — then monitors.

**Cost of a wrong call.** A false positive burns a review slot on a page that was already performing normally for its position; the editor may also "fix" a title that was fine and lose clicks. A false negative leaves a genuinely under-capturing page untouched for another cycle. Both costs are the editor's time and opportunity cost, not revenue I can measure — so the output is a prioritised review queue, not an instruction.

**Why data/ML helps.** The comparison is not eyeballable: judging whether a page under-captures requires knowing what "normal" looks like for its position tier and volume, across 30,000 pages. That is an aggregation problem before it is a modelling problem.


In [5]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# 1. How much of the inventory is even eligible? (visibility floor)
visible = df[df["impressions_90d"] >= 100].copy()
print(f"1) Pages with >=100 impressions: {len(visible):,} of {len(df):,} "
      f"({len(visible)/len(df):.0%}) — the reviewable universe.")

# 2. Median CTR differs sharply by position tier -> a flat CTR ranking would just rank position
print("\n2) Median CTR by position tier (visible pages only):")
print(visible.groupby("position_tier")["ctr"].agg(["median", "count"]).round(3).to_string())

# 3. How many pages sit below their own tier's median? -> the candidate pool
visible["expected_ctr"] = visible.groupby("position_tier")["ctr"].transform("median")
visible["ctr_gap"] = visible["ctr"] - visible["expected_ctr"]
below = (visible["ctr_gap"] < 0).sum()
print(f"\n3) Pages below their own tier's median CTR: {below:,} "
      f"({below/len(visible):.1%} of visible pages) — the pool this lane must rank.")


1) Pages with >=100 impressions: 22,006 of 30,000 (73%) — the reviewable universe.

2) Median CTR by position tier (visible pages only):
               median  count
position_tier               
deep             0.00    879
page_1           0.23   8633
page_3_5         0.06   6058
striking         0.15   5903
top_3            0.19    533

3) Pages below their own tier's median CTR: 10,307 (46.8% of visible pages) — the pool this lane must rank.


**What these numbers say.** Three-quarters of the inventory (22,006 pages) clears a 100-impression floor, so the lane has a real population to work with rather than a handful of pages. Median CTR varies roughly 4x across position tiers (0.23 on page_1 down to 0.00 for deep results), which confirms that any ranking on raw CTR would mostly re-rank position — the tier adjustment is not optional. And 10,307 pages (46.8% of visible pages) sit below their own tier's median, so a naive "below median" rule produces a candidate pool far larger than any review capacity. That gap between pool size and capacity is exactly what a scored, ordered queue is for.

**One thing I do not yet understand.** `top_3` shows a *lower* median CTR (0.19) than `page_1` (0.23), which is the opposite of what the position/CTR relationship predicts, and it is based on only 533 pages. Before building anything I need to check how these tiers are defined in `docs/data-dictionary.md` — if `page_1` and `top_3` overlap, or if `striking` sits between them, my grouping variable may need rework.

## 4. Careful words: what I can and can't claim

**What this work can say.** That, in this pseudonymised 30,000-page sample, certain pages *observed* a lower click-through rate than other pages sharing their position tier and volume band. That this gap is *measurable* and can be ordered, producing a prioritised list of review candidates. That the ranking is *decision-support* — a suggestion about where an editor should look first, backed by a stated reason code per page.

**What this work can never say.** That a low CTR gap was *caused* by a weak title or meta description; I observe an association, I run no experiment, and I cannot rule out intent mismatch, SERP features, brand familiarity, or seasonality. That rewriting a flagged page *will* increase clicks — I have no post-intervention measurement. That I am predicting or reverse-engineering Google's ranking behaviour; I am describing observed click behaviour in one sample, not modelling an algorithm. That results generalise beyond this dataset, this pseudonymisation, and this time window.

**Language I will hold myself to.** "observed", "measured", "associated with", "directional", "in this sample", "review candidate". Not "proves", "causes", "will improve", "Google ranks by".


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.